# K9 · Retrieval re-test after the re-ranking fix (no Gemini needed)
Fix: our SciBERT evidence finder now re-ranks **only paragraphs** (what it was trained on); figures and tables keep
the order from caption search + SigLIP image search. This re-runs the **retrieval part** of the system evaluation on the
**same questions** as K8 (same seed): QASPER Recall@1/3/5 + MRR, SPIQA Hit@1/3 + gold figure in evidence.
No answers are generated, so **no Gemini quota is used**.

Needs: **GPU T4 x2**, **Internet on**, secret **`HF_TOKEN`** (write). About 15–20 minutes.

## 0. Setup (Kaggle)
Before running: right panel → **Accelerator: GPU T4 x2 (or P100)**, **Internet: On** (needs a phone-verified Kaggle account),
and **Add-ons → Secrets → add `HF_TOKEN`** (your Hugging Face write token) and tick it for this notebook.

Kaggle has no Google Drive: results go to `/kaggle/working/ResearchLens` **and** every model + its results are pushed to your
private Hugging Face account. The last cell also backs up all results to a private HF dataset.

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/mkavinvelavan/ResearchLens.git"
DRIVE_ROOT = "/kaggle/working/ResearchLens"          # results (Kaggle's saved output folder)
SCRATCH = "/tmp/rl"                                    # data, checkpoints, final models (large, not saved)

os.environ["CUDA_VISIBLE_DEVICES"] = "0"               # one GPU, same setup as the Colab runs
for d in (DRIVE_ROOT, f"{SCRATCH}/content"):
    os.makedirs(d, exist_ok=True)
if not os.path.exists("/content"):                     # configs use /content/... paths (Colab layout)
    try:
        os.symlink(f"{SCRATCH}/content", "/content")
    except OSError as e:
        print("note: could not create /content link:", e)
os.makedirs("/content/data", exist_ok=True)

if not os.path.exists(f"{SCRATCH}/repo"):
    !git clone -q {REPO_URL} {SCRATCH}/repo
else:
    !git -C {SCRATCH}/repo pull -q
%cd {SCRATCH}/repo
!pip install -q -r requirements-colab.txt

os.environ.update({
    "RL_DRIVE_ROOT": DRIVE_ROOT, "RL_DATA_DIR": "/content/data",
    "RL_CKPT_DIR": f"{SCRATCH}/checkpoints", "RL_MODEL_DIR": f"{SCRATCH}/final_models",
    "TOKENIZERS_PARALLELISM": "false", "TQDM_DISABLE": "1",
})
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN found")
except Exception as e:
    print("WARNING: HF_TOKEN secret missing / not attached -> models will NOT be pushed to the Hub.", e)
!nvidia-smi --query-gpu=name,memory.total --format=csv

def run(module, config, run_name, seed, extra=""):
    # Train one (run, seed) unless it already finished; streams the log and stops on failure.
    import yaml
    tag = yaml.safe_load(open(config))["model_tag"]
    if os.path.exists(f"{DRIVE_ROOT}/results/{tag}/{run_name}/seed_{seed}/metrics.json"):
        print(f"skip {run_name} seed {seed} (finished)"); return
    if "--push_to_hub" in extra and not os.environ.get("HF_TOKEN"):
        extra = extra.replace("--push_to_hub", "")
    cmd = f"{sys.executable} -m {module} --config {config} --run_name {run_name} --seed {seed} {extra}"
    print(">>>", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in p.stdout:
        print(line, end="", flush=True)
        tail = (tail + [line])[-40:]
    if p.wait() != 0:
        raise RuntimeError(f"Training failed: {run_name} seed {seed}. Last lines:\n" + "".join(tail))

## 1. Install the app's packages

In [ ]:
!pip install -q -r requirements-app.txt

## 2. Retrieval-only evaluation (same 100 QASPER + 50 SPIQA questions as K8)

In [ ]:
OUT = "/kaggle/working/system_eval_v2"
!python -m researchlens.evaluation.system_eval --out_dir {OUT} --data_dir /kaggle/working/eval_data --n_qasper 100 --n_spiqa 50 --no_answers

## 3. Results + upload to `results/system_eval_v2/`

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(f"{OUT}/report.md").read()))
from huggingface_hub import HfApi
HfApi().upload_folder(repo_id="Kavinvelavan/researchlens-results", repo_type="dataset", folder_path=OUT,
                      path_in_repo="results/system_eval_v2", ignore_patterns=["qdrant/*", "papers/*"],
                      token=os.environ.get("HF_TOKEN"), commit_message="retrieval re-test after re-ranking fix")
print("uploaded")